# Build the CertVIC CPython 3.12 offline wheelhouse

Settings: **Accelerator OFF**, **Internet ON**. Attach the refreshed authenticated CertVIC CODE, CONFIGS, and EXECUTION_TOOLS inputs. Run All without editing. A failed or partial build emits only `certvic_cp312_wheelhouse_failure_report.json`; only a fully verified run produces `certvic_offline_wheelhouse_cp312.zip`.


In [ ]:
import json, os, pathlib, platform, shutil, sys
from packaging.tags import sys_tags

probe = {
    "executable": sys.executable,
    "implementation": platform.python_implementation(),
    "python": platform.python_version(),
    "architecture": platform.machine(),
    "system": platform.system(),
    "libc": platform.libc_ver(),
    "supported_tags": [str(tag) for tag in sys_tags()],
}
print(json.dumps({"status": "IMMEDIATE_CP312_PROVISIONING_PROBE", **probe}, indent=2))
if (probe["implementation"] != "CPython" or not probe["python"].startswith("3.12.")
        or probe["architecture"].lower() != "x86_64" or probe["system"] != "Linux"):
    raise RuntimeError(
        "CERTVIC_RUNTIME_01_PYTHON_PROFILE_NOT_SUPPORTED: "
        "builder requires Kaggle CPython 3.12 Linux x86_64"
    )
if (not probe["libc"][0].lower().startswith("glibc")
        or tuple(map(int, probe["libc"][1].split("."))) < (2, 17)):
    raise RuntimeError(
        "CERTVIC_RUNTIME_01_PYTHON_PROFILE_NOT_SUPPORTED: glibc >= 2.17 required"
    )


In [ ]:
import hashlib, json, os, pathlib, shutil, stat, sys, zipfile

DISCOVERY_ERRORS = {
    "missing": "CERTVIC_DISCOVERY_01_REQUIRED_ROLE_NOT_FOUND",
    "ambiguous": "CERTVIC_DISCOVERY_02_AMBIGUOUS_DISTINCT_CONTENT",
    "authentication": "CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED",
}
DISCOVERY_POLICY = "CONTENT_AUTHENTICATED_ANY_LOCATION"
OPERATIONAL_FIELDS = {
    "builder_command", "created_time", "expected_kaggle_dataset_slug", "mount_path",
    "required_notebook", "validation_command",
}

def early_sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def early_safe_member(info):
    name = info.filename
    normalized = name.replace("\\", "/")
    value = pathlib.PurePosixPath(normalized)
    mode = (info.external_attr >> 16) & 0xFFFF
    if (not normalized or normalized != name or normalized.endswith("/") or value.is_absolute()
            or ".." in value.parts or "." in value.parts or normalized.startswith("~")
            or "\x00" in normalized or info.is_dir() or stat.S_ISLNK(mode)
            or (mode and not stat.S_ISREG(mode))):
        raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe member {name!r}")
    return value.as_posix()

def early_content_identity(manifest, hash_files):
    identity_manifest = {key: value for key, value in manifest.items()
                         if key not in OPERATIONAL_FIELDS}
    identity_files = {name: record for name, record in hash_files.items()
                      if name not in {"README.md", "bundle_manifest.json"}}
    payload = json.dumps({"manifest": identity_manifest, "files": identity_files},
                         indent=2, sort_keys=True).encode() + b"\n"
    return hashlib.sha256(payload).hexdigest()

def early_verify_archive(path):
    with zipfile.ZipFile(path) as archive:
        infos = archive.infolist()
        names = [early_safe_member(info) for info in infos]
        if "bundle_manifest.json" not in names or "hash_manifest.json" not in names:
            return None
        if len(names) != len(set(names)) or archive.testzip() is not None:
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: duplicate or corrupt archive")
        manifest_bytes = archive.read("bundle_manifest.json")
        hash_bytes = archive.read("hash_manifest.json")
        if len(manifest_bytes) > 8 * 1024 * 1024 or len(hash_bytes) > 8 * 1024 * 1024:
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: oversized manifest")
        manifest = json.loads(manifest_bytes)
        hashes = json.loads(hash_bytes)
        if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
                or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"
                or manifest.get("bundle_type") != "CODE"):
            return None
        declared, hash_files = manifest.get("files", {}), hashes.get("files", {})
        if (set(names) != set(hash_files) | {"hash_manifest.json"}
                or set(declared) != set(names) - {"bundle_manifest.json", "hash_manifest.json"}):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: file universe mismatch")
        for name, record in hash_files.items():
            payload = archive.read(name)
            observed = {"size": len(payload), "sha256": hashlib.sha256(payload).hexdigest()}
            if record != observed or (name in declared and declared[name] != observed):
                raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: byte mismatch {name}")
        return manifest, hash_files, hashlib.sha256(manifest_bytes).hexdigest()

def early_verify_directory(path):
    root = pathlib.Path(path).resolve()
    manifest_path, hash_path = root / "bundle_manifest.json", root / "hash_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    hashes = json.loads(hash_path.read_text(encoding="utf-8"))
    if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
            or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"
            or manifest.get("bundle_type") != "CODE"):
        return None
    observed = {}
    for member in root.rglob("*"):
        mode = member.lstat().st_mode
        if member.is_symlink() or not (stat.S_ISDIR(mode) or stat.S_ISREG(mode)):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe extracted member")
        if stat.S_ISREG(mode):
            observed[member.relative_to(root).as_posix()] = member
    declared, hash_files = manifest.get("files", {}), hashes.get("files", {})
    if (set(observed) != set(hash_files) | {"hash_manifest.json"}
            or set(declared) != set(observed) - {"bundle_manifest.json", "hash_manifest.json"}):
        raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: extracted universe mismatch")
    for name, record in hash_files.items():
        member = observed[name]
        actual = {"size": member.stat().st_size, "sha256": early_sha256(member)}
        if actual != record or (name in declared and declared[name] != actual):
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: extracted byte mismatch {name}")
    return manifest, hash_files, early_sha256(manifest_path)

configured_roots = [value for value in os.environ.get("CERTVIC_INPUT_ROOTS", "").split(os.pathsep)
                    if value]
if not configured_roots:
    configured_roots = ["/kaggle/input", "/kaggle/working"]
INPUT_ROOTS = sorted({str(pathlib.Path(value).resolve()) for value in configured_roots
                      if pathlib.Path(value).is_dir() and not pathlib.Path(value).is_symlink()})
archive_candidates, directory_candidates = [], []
for root_value in INPUT_ROOTS:
    root = pathlib.Path(root_value)
    for current, directory_names, file_names in os.walk(root, followlinks=False):
        base = pathlib.Path(current)
        directory_names[:] = sorted(name for name in directory_names
                                     if not (base / name).is_symlink())
        if "bundle_manifest.json" in file_names and "hash_manifest.json" in file_names:
            directory_candidates.append(base.resolve())
        for name in sorted(file_names):
            candidate = base / name
            if candidate.is_symlink() or not candidate.is_file():
                continue
            try:
                with candidate.open("rb") as handle:
                    magic = handle.read(4)
            except OSError:
                continue
            if magic in {b"PK\x03\x04", b"PK\x05\x06", b"PK\x07\x08"}:
                archive_candidates.append(candidate.resolve())

valid, failures = [], []
for representation, candidates in (("zip_archive", sorted(set(archive_candidates))),
                                   ("extracted_directory", sorted(set(directory_candidates)))):
    for candidate in candidates:
        try:
            result = (early_verify_archive(candidate) if representation == "zip_archive"
                      else early_verify_directory(candidate))
        except (OSError, KeyError, json.JSONDecodeError, UnicodeDecodeError,
                zipfile.BadZipFile, RuntimeError) as error:
            failures.append(f"{candidate}: {error}")
            continue
        if result is None:
            continue
        manifest, hash_files, manifest_hash = result
        identity = early_content_identity(manifest, hash_files)
        expected = os.environ.get("CERTVIC_EXPECTED_CONTENT_ID_CODE")
        if expected and identity != expected.lower():
            failures.append(f"{candidate}: expected CODE content identity mismatch")
            continue
        valid.append({"path": candidate, "representation": representation,
                      "manifest": manifest, "manifest_sha256": manifest_hash,
                      "content_identity_sha256": identity})
if not valid:
    code = DISCOVERY_ERRORS["authentication"] if failures else DISCOVERY_ERRORS["missing"]
    raise RuntimeError(f"{code}: role=CODE failures={failures}")
identities = {row["content_identity_sha256"] for row in valid}
if len(identities) != 1:
    raise RuntimeError(f"{DISCOVERY_ERRORS['ambiguous']}: role=CODE candidates="
                       f"{[(row['content_identity_sha256'], str(row['path'])) for row in valid]}")
selected = min(valid, key=lambda row: os.path.normcase(str(row["path"])))
CODE_DISCOVERY_MIRRORS = sorted({str(row["path"]) for row in valid})
CODE_BUNDLE_SOURCE = str(selected["path"])
CODE_BUNDLE_HASH = selected["content_identity_sha256"]
CODE_ARCHIVE_SHA256 = (early_sha256(selected["path"])
                       if selected["representation"] == "zip_archive" else None)
if selected["representation"] == "zip_archive":
    CODE_EXTRACT_ROOT = pathlib.Path(os.environ.get(
        "CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")) / "certvic_code"
    if CODE_EXTRACT_ROOT.exists():
        if CODE_EXTRACT_ROOT.is_symlink() or not CODE_EXTRACT_ROOT.is_dir():
            raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: unsafe CODE destination")
        shutil.rmtree(CODE_EXTRACT_ROOT)
    CODE_EXTRACT_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(selected["path"]) as archive:
        for info in archive.infolist():
            name = early_safe_member(info)
            output = (CODE_EXTRACT_ROOT / name).resolve()
            output.relative_to(CODE_EXTRACT_ROOT.resolve())
            output.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as reader, output.open("xb") as writer:
                shutil.copyfileobj(reader, writer, length=1024 * 1024)
else:
    CODE_EXTRACT_ROOT = pathlib.Path(selected["path"])
CODE_BUNDLE_PATH = (CODE_BUNDLE_SOURCE if selected["representation"] == "zip_archive"
                    else str(CODE_EXTRACT_ROOT / "bundle_manifest.json"))
CODE_BUNDLE = CODE_BUNDLE_PATH
project_candidates = sorted(path.parent.resolve() for path in CODE_EXTRACT_ROOT.rglob("pyproject.toml")
                            if (path.parent / "certvic/__init__.py").is_file())
if len(project_candidates) != 1:
    raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: CODE project root ambiguous")
PROJECT_ROOT = project_candidates[0]
sys.path.insert(0, str(PROJECT_ROOT))
from certvic.cvpr.content_discovery import (
    DISCOVERY_POLICY, discover_authenticated_input, resolve_content_bound_roles,
)
from certvic.cvpr.notebook_bootstrap import discover_unique_file, discover_unique_root
authenticated_code = discover_authenticated_input(
    "CODE", roots=INPUT_ROOTS, expected_identity=CODE_BUNDLE_HASH,
    materialization_root=pathlib.Path(os.environ.get(
        "CERTVIC_KAGGLE_WORKING_ROOT", "/kaggle/working")) / "certvic_authenticated_inputs",
)
if authenticated_code["content_identity_sha256"] != CODE_BUNDLE_HASH:
    raise RuntimeError(f"{DISCOVERY_ERRORS['authentication']}: early/shared CODE identity mismatch")
AUTHENTICATED_CONTENT_IDENTITIES = {"code_bundle": CODE_BUNDLE_HASH}
DISCOVERED_PROVENANCE = {"CODE": authenticated_code}
print({"discovery_policy": DISCOVERY_POLICY, "role": "CODE", "provider": None,
       "study": selected["manifest"].get("study"), "stage": selected["manifest"].get("stage"),
       "representation": selected["representation"], "discovered_path": CODE_BUNDLE_SOURCE,
       "content_identity_sha256": CODE_BUNDLE_HASH, "archive_sha256": CODE_ARCHIVE_SHA256,
       "mirrors": CODE_DISCOVERY_MIRRORS, "project_root": str(PROJECT_ROOT)})


In [ ]:
from certvic.cvpr.content_discovery import discover_authenticated_input
from certvic.cvpr.notebook_bootstrap import discover_unique_file
from certvic.cvpr.wheelhouse_builder import WheelhouseBuilderError, deterministic_provision

materialized = pathlib.Path("/kaggle/working/certvic_provisioning_inputs")
code = discover_authenticated_input(
    "CODE", roots=INPUT_ROOTS, expected_identity=CODE_BUNDLE_HASH,
    materialization_root=materialized,
)
configs = discover_authenticated_input(
    "CONFIGS", roots=INPUT_ROOTS, materialization_root=materialized,
)
tools = discover_authenticated_input(
    "EXECUTION_TOOLS", roots=INPUT_ROOTS, materialization_root=materialized,
)
print({
    "authenticated_content_identities": {
        row["role"]: row["content_identity_sha256"] for row in (code, configs, tools)
    }
})
environment_lock = discover_unique_file(
    configs["materialized_root"], "kaggle_t4x2_environment.lock.json"
)
requirements_root = discover_unique_file(
    configs["materialized_root"], "kaggle_base.lock"
).parent
wheel_root = pathlib.Path("/kaggle/working/certvic_cp312_wheels")
output = pathlib.Path("/kaggle/working/certvic_offline_wheelhouse_cp312.zip")
failure_report_path = pathlib.Path(
    "/kaggle/working/certvic_cp312_wheelhouse_failure_report.json"
)

def failure_names(report):
    names = set()
    for field in ("incompatible_wheels", "missing_packages", "duplicate_conflicts"):
        for value in report.get(field, []):
            if isinstance(value, dict):
                names.add(str(value.get("package") or value.get("filename") or value))
            else:
                names.add(str(value))
    return sorted(names)

try:
    result = deterministic_provision(
        wheel_root=wheel_root,
        output=output,
        requirements_root=requirements_root,
        profile_id="kaggle_cp312_2026_07",
        environment_lock=environment_lock,
        failure_report_path=failure_report_path,
    )
except WheelhouseBuilderError as error:
    report = dict(error.report)
    if failure_report_path.is_file():
        report = json.loads(failure_report_path.read_text(encoding="utf-8"))
    print(json.dumps(report, indent=2, sort_keys=True))
    raise RuntimeError(
        f"{report.get('status', error.status)}: packages={failure_names(report)}; "
        f"failure_report={failure_report_path}"
    ) from error
if not result.get("passed", False):
    report = dict(result)
    print(json.dumps(report, indent=2, sort_keys=True))
    raise RuntimeError(
        f"{report.get('status')}: packages={failure_names(report)}; "
        f"failure_report={failure_report_path}"
    )
print(json.dumps(result, indent=2, sort_keys=True))


In [ ]:
from certvic.cvpr.environment_lock import (
    EnvironmentLockError,
    HOST_PIP_CANNOT_TARGET_VENV,
    prepare_offline_environment,
    select_locked_runtime,
)
from certvic.cvpr.kaggle_bundle import verify_bundle
from certvic.cvpr.notebook_bootstrap import extract_verified_bundle
from certvic.cvpr.wheelhouse_builder import (
    persist_failure_report,
    provisioning_failure_report,
)

selected_profile = select_locked_runtime(environment_lock)
try:
    if not output.is_file() or result.get("deterministic_rebuild", {}).get(
        "byte_identical"
    ) is not True:
        raise RuntimeError("passed deterministic bundle was not produced")
    verification = verify_bundle(output)
    if not verification["passed"]:
        raise RuntimeError(f"authenticated bundle verification failed: {verification['errors']}")
    validation_root = pathlib.Path("/kaggle/working/certvic_cp312_offline_validation")
    extract_verified_bundle(
        output,
        validation_root,
        expected_type="OFFLINE_LINUX_WHEELHOUSE",
    )
    manifest = json.loads(
        (validation_root / "wheelhouse_manifest.json").read_text(encoding="utf-8")
    )
    offline_validation = prepare_offline_environment(
        environment_lock,
        wheelhouse=validation_root / "wheels",
        wheelhouse_manifest=validation_root / "wheelhouse_manifest.json",
        allow_preinstalled=False,
        require_exact=True,
        require_cuda=False,
        selected_profile=selected_profile,
        venv_root=pathlib.Path(
            "/kaggle/working/certvic_runtime/kaggle_cp312_builder_validation"
        ),
    )
except EnvironmentLockError as error:
    report = dict(error.report)
    persist_failure_report(failure_report_path, report)
    output.unlink(missing_ok=True)
    print(json.dumps(report, indent=2, sort_keys=True))
    raise RuntimeError(
        f"{report['status']}: offline_validation; failure_report={failure_report_path}"
    ) from error
except Exception as error:
    report = provisioning_failure_report(
        "CERTVIC_RUNTIME_09_OFFLINE_VALIDATION_FAILED",
        selected=selected_profile,
        required_packages=result.get("resolver_result", {}).get("required_packages", {}),
        provisioning=result.get("resolver_result", {}),
        downloaded_wheels=list(locals().get("manifest", {}).get("files", {}).values()),
        remediation=(
            "Inspect the named offline install/import failure. Do not upload the bundle until "
            "a clean ensurepip-free CPython 3.12 venv targeted by host pip --python passes every import."
        ),
    )
    report["offline_validation_error"] = f"{type(error).__name__}: {error}"
    if HOST_PIP_CANNOT_TARGET_VENV in str(error):
        report["status"] = HOST_PIP_CANNOT_TARGET_VENV
    persist_failure_report(failure_report_path, report)
    output.unlink(missing_ok=True)
    print(json.dumps(report, indent=2, sort_keys=True))
    raise RuntimeError(
        f"{report['status']}: offline_validation; failure_report={failure_report_path}"
    ) from error
print({
    "resolver_result": result.get("resolver_result"),
    "supported_tags": selected_profile["observed_runtime"]["supported_tags"],
    "wheel_hashes": {
        name: row["sha256"]
        for name, row in offline_validation["wheelhouse_validation"]["files"].items()
    },
    "offline_install_import_validation": offline_validation,
})
print({
    "status": "CP312_WHEELHOUSE_BUILDER_READY",
    "runtime_profile": "kaggle_cp312_2026_07",
    "bundle_sha256": verification["sha256"],
    "size": output.stat().st_size,
    "wheel_count": result.get("wheel_count"),
    "deterministic_rebuild": result["deterministic_rebuild"],
    "offline_validation_status": offline_validation["status"],
    "ensurepip_used": offline_validation.get("ensurepip_used", False),
    "kernel_packages_mutated": offline_validation.get("kernel_packages_mutated", False),
    "network_used_for_provisioning": True,
    "paper_evidence": False,
})
print(
    "NEXT: download certvic_offline_wheelhouse_cp312.zip, import it unchanged with "
    "kagglefiles/import_kaggle_return.py, then run 00A with Accelerator OFF and Internet OFF."
)
